# Exercise 39 — `yield` and generator functions

## Concept

Any function containing `yield` becomes a **generator function**. Calling it doesn't run the body — it returns a generator object. Each `next(...)` call (explicit, or via a `for` loop) runs until the next `yield`, then pauses, remembering local state.

```python
def counter(n):
    i = 0
    while i < n:
        yield i
        i += 1

for x in counter(3):
    print(x)        # 0, 1, 2
```

Why this matters: you can produce values **as you need them**, without holding the full sequence in memory. Same memory benefit as a generator expression, but with the full power of statements (loops, conditions, multi-line transforms).

A generator naturally finishes when the function returns (or just runs off the end).

### Common DE pattern: streaming line-by-line

```python
def iter_log_lines(path):
    with open(path) as f:
        for line in f:
            yield line.rstrip("\n")
```

The caller iterates millions of lines using `O(1)` memory.

## Task 1 — Reimplement `range`-ish

Write a generator function `count_up(start, stop, step=1)` that yields integers from `start` (inclusive) up to but not including `stop`. Use `yield` — not `range`.

```python
def count_up(start: int, stop: int, step: int = 1):
    ...
```

Expected:
- `list(count_up(0, 5))` → `[0, 1, 2, 3, 4]`
- `list(count_up(2, 10, 3))` → `[2, 5, 8]`
- `list(count_up(5, 5))` → `[]`

Note: don't worry about negative steps for this task.

In [ ]:
def count_up(start: int, stop: int, step: int = 1):
    pass  # TODO

assert list(count_up(0, 5)) == [0, 1, 2, 3, 4]
assert list(count_up(2, 10, 3)) == [2, 5, 8]
assert list(count_up(5, 5)) == []
print("ok")


## Task 2 — Filter a stream

Write a generator `only_errors(lines)` that consumes an iterable of log strings and yields only those starting with `"ERROR"`.

```python
def only_errors(lines):
    ...
```

Why a generator and not a function returning a list? Because the input could itself be a generator over a giant file — and we want to keep memory flat. **Don't** build a list inside the function.

Expected:
- `list(only_errors(["INFO a", "ERROR b", "WARN c", "ERROR d"]))` → `["ERROR b", "ERROR d"]`
- `list(only_errors([]))` → `[]`

In [ ]:
def only_errors(lines):
    pass  # TODO

assert list(only_errors(["INFO a", "ERROR b", "WARN c", "ERROR d"])) == ["ERROR b", "ERROR d"]
assert list(only_errors([])) == []
print("ok")


## Task 3 — Batch a stream

Write a generator `batched(items, size)` that yields **lists** of up to `size` items at a time. The last batch may be smaller.

```python
def batched(items, size: int):
    ...
```

Expected:
- `list(batched([1, 2, 3, 4, 5, 6, 7], 3))` → `[[1, 2, 3], [4, 5, 6], [7]]`
- `list(batched([], 3))` → `[]`
- `list(batched([1, 2], 5))` → `[[1, 2]]`

Why this pattern: bulk inserts to a DB or HTTP API usually want N rows per request. Naive `for row in stream: insert(row)` is slow; batching is much faster.

Hint: maintain a buffer list. `yield` it whenever it reaches `size`, then reset. **After the loop**, yield any leftover items if the buffer isn't empty.

In [ ]:
def batched(items, size: int):
    pass  # TODO

assert list(batched([1, 2, 3, 4, 5, 6, 7], 3)) == [[1, 2, 3], [4, 5, 6], [7]]
assert list(batched([], 3)) == []
assert list(batched([1, 2], 5)) == [[1, 2]]
assert list(batched(range(6), 2)) == [[0, 1], [2, 3], [4, 5]]
print("ok")


## Task 4 — Chain generators (pipeline)

Using the generators from tasks 1–3, build a pipeline that:
1. Generates the integers 0..19 (using `count_up`)
2. Yields only the **even** ones — write a small generator `keep_even(nums)` here
3. Batches them by 3

Assemble the pipeline and convert the final result to a list to inspect it.

Expected:
```
[[0, 2, 4], [6, 8, 10], [12, 14, 16], [18]]
```

Note how each stage is its own small generator. No intermediate list is built — values flow through one at a time.

In [ ]:
def keep_even(nums):
    pass  # TODO

# Build the pipeline and assert
result = list(batched(keep_even(count_up(0, 20)), 3))
assert result == [[0, 2, 4], [6, 8, 10], [12, 14, 16], [18]]
print("ok")
